In [ ]:
# --- Minimal CBOW Word Embeddings (Keras) ---

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1) Toy corpus
corpus = [
    "The cat sat on the mat",
    "The dog ran in the park",
    "The bird sang in the tree",
]

# 2) Tokenize → integer sequences
tok = Tokenizer()                 # lowercases by default
tok.fit_on_texts(corpus)
seqs = tok.texts_to_sequences(corpus)
vocab_size = len(tok.word_index) + 1  # +1 for padding index 0

# 3) Build CBOW training pairs (context words → target word)
window = 2
contexts, targets = [], []
for s in seqs:
    for i in range(window, len(s) - window):
        ctx = s[i - window:i] + s[i + 1:i + window + 1]  # 2 left + 2 right
        tgt = s[i]
        contexts.append(ctx)
        targets.append(tgt)

X = np.array(contexts)                               # shape: (num_samples, 2*window)
y = to_categorical(targets, num_classes=vocab_size)  # one-hot targets

# 4) Tiny CBOW model: Embedding → mean → softmax
embed_dim = 10
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embed_dim),  # (batch, 2*window, emb)
    Lambda(lambda t: tf.reduce_mean(t, axis=1)),            # (batch, emb)
    Dense(vocab_size, activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=100, verbose=0)

# 5) Extract learned word vectors
W = model.layers[0].get_weights()[0]  # (vocab_size, embed_dim)

# 6) 2D visualize with PCA
reduced = PCA(n_components=2).fit_transform(W)

plt.figure(figsize=(5, 5))
for word, idx in tok.word_index.items():
    x, y2 = reduced[idx]  # idx 0 is padding; our words start at 1
    plt.scatter(x, y2)
    plt.annotate(word, (x, y2))
plt.title("CBOW Word Embeddings (2D PCA)")
plt.show()

vocab_size = len(tokenizer.word_index) + 1
embedding_size = 10
window_size = 2

contexts = []
targets = []
for sequence in sequences:
    for i in range(window_size, len(sequence) - window_size):
        context = sequence[i - window_size:i] + sequence[i + 1:i + window_size + 1]
        target = sequence[i]
        contexts.append(context)
        targets.append(target)

X = np.array(contexts)
y = to_categorical(targets, num_classes=vocab_size)

print(X,y)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_size),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X, y, epochs=100, verbose=0)

model.summary()


embeddings = model.layers[0].get_weights()[0]

pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)

plt.figure(figsize=(5,5))
for word, idx in tokenizer.word_index.items():
    x, y = reduced_embeddings[idx]
    plt.scatter(x, y)
    plt.annotate(word, xy=(x,y))
plt.title("Word Embeddings Visualized")
plt.show()

ERROR: Could not find a version that satisfies the requirement tesnorflow==2.11.0 (from versions: none)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tesnorflow==2.11.0
